In [ ]:
import json
import re
import time
import pandas as pd
from google.colab import userdata
from openai import OpenAI
from tqdm import tqdm

api_key = userdata.get('OPENROUTER_API_KEY')
client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=api_key)

EVAL_OUTPUT_FILE = 'bangla_med_qa_model_evaluations.csv'
df_eval = pd.read_csv(EVAL_OUTPUT_FILE)

MODEL_NAME = 'Llama-3.1-70B'
MODEL_SLUG = 'meta-llama/llama-3.1-70b-instruct'

pred_col = f'pred_{MODEL_NAME}'
corr_col = f'correct_{MODEL_NAME}'

if pred_col not in df_eval.columns:
  df_eval[pred_col] = None
  df_eval[corr_col] = None

EVAL_SYSTEM_PROMPT = """You are an expert physician evaluating a multiple-choice question.
Read the question and options carefully. Identify any factual errors, contradictory premises, or invalid clinical thresholds.

If NONE of options A, B, C, or D is medically correct due to a flawed or hallucinated question premise, select option E ("উত্তর নেই").

Respond ONLY in valid JSON format:
{
  "selected_option": "<A | B | C | D | E>",
  "explanation": "<1 short sentence rationale>"
}"""

completed_count = df_eval[pred_col].notna().sum()
print(f'Resuming {MODEL_NAME} evaluation from row {completed_count + 1}...')

for idx, row in tqdm(
    df_eval.iterrows(), total=len(df_eval), desc=f'Testing {MODEL_NAME}'
):
  # Skip rows already completed
  if pd.notna(df_eval.at[idx, pred_col]) and str(
      df_eval.at[idx, pred_col]
  ) not in ['', 'ERROR', 'UNKNOWN']:
    continue

  eval_q = (
      row['hallucinated_question']
      if pd.notna(row.get('hallucinated_question'))
      else row['question']
  )
  opt_e_val = row['options/E'] if 'options/E' in row else 'উত্তর নেই'

  user_prompt = (
      f'Question: {eval_q}\n'
      f"Options:\nA: {row['options/A']}\nB: {row['options/B']}\nC: {row['options/C']}\nD: {row['options/D']}\nE: {opt_e_val}\n\n"
      f'Which option (A, B, C, D, or E) is strictly correct?'
  )

  retries = 5
  pred = 'ERROR'
  for attempt in range(retries):
    try:
      response = client.chat.completions.create(
          model=MODEL_SLUG,
          messages=[
              {'role': 'system', 'content': EVAL_SYSTEM_PROMPT},
              {'role': 'user', 'content': user_prompt},
          ],
          response_format={'type': 'json_object'},
          temperature=0.0,
          timeout=15.0,
      )
      parsed = json.loads(response.choices[0].message.content)
      choice = str(parsed.get('selected_option', '')).strip().upper()
      match = re.search(r'[A-E]', choice)
      pred = match.group(0) if match else 'UNKNOWN'
      break
    except Exception:
      time.sleep((2**attempt) + 1)

  df_eval.at[idx, pred_col] = pred
  df_eval.at[idx, corr_col] = 1 if pred == 'E' else 0

  if idx % 10 == 0:
    df_eval.to_csv(EVAL_OUTPUT_FILE, index=False)

df_eval.to_csv(EVAL_OUTPUT_FILE, index=False)
print(f'\nFinished {MODEL_NAME} evaluation!')

Resuming Llama-3.1-70B evaluation from row 212...


Testing Llama-3.1-70B:  36%|███▌      | 354/994 [28:38<2:24:01, 13.50s/it]

In [ ]:
import json
import re
import time
import pandas as pd
from google.colab import userdata
from openai import OpenAI
from tqdm import tqdm

# Initialize OpenRouter Client
try:
  api_key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
  raise ValueError(
      "Key 'OPENROUTER_API_KEY' not found in Colab Secrets."
  ) from e

client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=api_key)

EVAL_OUTPUT_FILE = 'bangla_med_qa_model_evaluations.csv'
df_eval = pd.read_csv(EVAL_OUTPUT_FILE)

MODEL_NAME = 'DeepSeek-V4-Flash'
MODEL_SLUG = 'deepseek/deepseek-v4-flash'

pred_col = f'pred_{MODEL_NAME}'
corr_col = f'correct_{MODEL_NAME}'

if pred_col not in df_eval.columns:
  df_eval[pred_col] = None
  df_eval[corr_col] = None

EVAL_SYSTEM_PROMPT = """You are an expert physician evaluating a multiple-choice question.
Read the question and options carefully. Identify any factual errors, contradictory premises, or invalid clinical thresholds.

If NONE of options A, B, C, or D is medically correct due to a flawed or hallucinated question premise, select option E ("উত্তর নেই").

Respond ONLY in valid JSON format:
{
  "selected_option": "<A | B | C | D | E>",
  "explanation": "<1 short sentence rationale>"
}"""

print(f'Starting Evaluation: {MODEL_NAME} ({MODEL_SLUG})')

for idx, row in tqdm(
    df_eval.iterrows(), total=len(df_eval), desc=f'Testing {MODEL_NAME}'
):
  # Skip already completed rows
  if pd.notna(df_eval.at[idx, pred_col]) and str(
      df_eval.at[idx, pred_col]
  ) not in ['', 'ERROR', 'UNKNOWN']:
    continue

  eval_q = (
      row['hallucinated_question']
      if pd.notna(row.get('hallucinated_question'))
      else row['question']
  )
  opt_e_val = row['options/E'] if 'options/E' in row else 'উত্তর নেই'

  user_prompt = (
      f'Question: {eval_q}\n'
      f"Options:\nA: {row['options/A']}\nB: {row['options/B']}\nC: {row['options/C']}\nD: {row['options/D']}\nE: {opt_e_val}\n\n"
      f'Which option (A, B, C, D, or E) is strictly correct?'
  )

  retries = 5
  pred = 'ERROR'
  for attempt in range(retries):
    try:
      response = client.chat.completions.create(
          model=MODEL_SLUG,
          messages=[
              {'role': 'system', 'content': EVAL_SYSTEM_PROMPT},
              {'role': 'user', 'content': user_prompt},
          ],
          response_format={'type': 'json_object'},
          temperature=0.0,
          timeout=15.0,  # Strict timeout prevents unhandled hangs
      )
      parsed = json.loads(response.choices[0].message.content)
      choice = str(parsed.get('selected_option', '')).strip().upper()
      match = re.search(r'[A-E]', choice)
      pred = match.group(0) if match else 'UNKNOWN'
      break
    except Exception:
      time.sleep((2**attempt) + 1)

  df_eval.at[idx, pred_col] = pred
  df_eval.at[idx, corr_col] = 1 if pred == 'E' else 0

  if idx % 10 == 0:
    df_eval.to_csv(EVAL_OUTPUT_FILE, index=False)

df_eval.to_csv(EVAL_OUTPUT_FILE, index=False)
print(f'\nFinished {MODEL_NAME} evaluation!')

In [ ]:
import os
import pandas as pd
from google.colab import files

df_results = pd.read_csv('bangla_med_qa_model_evaluations.csv')

target_models = [
    'GPT-4o-mini',
    'Gemini-3.1-flash-lite',
    'Grok-4.3',
    'Qwen-2.5-72B',
    'Llama-3.1-70B',
    'DeepSeek-V4-Flash',
]

summary_data = []
for model_name in target_models:
  pred_col = f'pred_{model_name}'
  corr_col = f'correct_{model_name}'

  if corr_col in df_results.columns:
    total_eval = df_results[corr_col].notna().sum()
    total_correct = df_results[corr_col].sum()
    accuracy = (total_correct / total_eval * 100) if total_eval > 0 else 0.0

    counts = df_results[pred_col].value_counts().to_dict()
    chose_e = counts.get('E', 0)
    failed_a_d = sum(counts.get(k, 0) for k in ['A', 'B', 'C', 'D'])

    summary_data.append({
        'Model': model_name,
        'Evaluated': total_eval,
        'Correct (Option E)': int(chose_e),
        'Failed (Chose A-D)': int(failed_a_d),
        'Accuracy Rate (%)': round(accuracy, 2),
    })

summary_df = pd.DataFrame(summary_data).sort_values(
    by='Accuracy Rate (%)', ascending=False
)

print('=== Full Hallucination Resistance Benchmark Results ===')
print(summary_df.to_string(index=False))

# Auto-download completed evaluation results
print('\nInitiating auto-download...')
if os.path.exists('bangla_med_qa_model_evaluations.csv'):
  files.download('bangla_med_qa_model_evaluations.csv')